In [1]:
from langchain_core.documents import Document
from semantic_text_splitter import TextSplitter
from langchain_openai import OpenAIEmbeddings

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

import json
from pathlib import Path
from typing import List, Any, Dict, Tuple
import shutil,stat

# from concurrent.futures import ThreadPoolExecutor, as_completed
# from semantic_chunker import get_chunker as SemanticChunker

/Users/tanweihui/Downloads/RAG-for-querying/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
documents = []

for file_path in Path("../data/").glob("*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        articles = json.load(f)
        
    
    for article in articles["data"]["results"]:
        documents.append(
            Document(
                page_content=article["content"],
                metadata={
                    "id":           article["id"],
                    "title":        article["title"],
                    "source":       article["source"],
                    "link":         str(article["link"]),
                    "publish_date": str(article["publish_date"]),
                    "images":       str(article.get("images")),
                }
            )
        )

print(f"Loaded {len(documents)} articles from {len(list(Path('../data/').glob('*.json')))} files")

Loaded 440 articles from 11 files


In [4]:
print(documents[0].metadata)
print(documents[0].page_content[:500])

{'id': 'c0a315ff3558239f19d0482261befcfa6fff9359b5e47eeaff1630ef5d8723b5', 'title': 'Iceland Joins Switzerland, Turkey, Ireland, Denmark (Greenland), Colombia, and Italy as the Must-Visit Destinations for US Travelers in 2026, Offering Stunning Landscapes and Unique Adventures - Travel And Tour World', 'source': 'feeds.feedburner.com', 'link': 'https://www.travelandtourworld.com/news/article/iceland-joins-switzerland-turkey-ireland-denmark-greenland-colombia-and-italy-as-the-must-visit-destinations-for-us-travelers-in-2026-offering-stunning-landscapes-and-unique-adventures/', 'publish_date': '2025-12-20T08:00:00+08:00', 'images': "['https://www.travelandtourworld.com/wp-content/uploads/2025/12/iceland-joins-switzerland-turkey-ireland_j_GaAnXzQI2f3ZNahd3b_Q_EvSbLFoeR8S583HtKh2ldg.jpg']"}
Home»America Travel News» Iceland Joins Switzerland, Turkey, Ireland, Denmark (Greenland), Colombia, and Italy as the Must-Visit Destinations for US Travelers in 2026, Offering Stunning Landscapes and U

In [ ]:
from tokenizers import Tokenizer

class EmbeddingDocument:
    def __init__(self, capacity: int = 256):
        tokenizer = Tokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
        self.splitter = TextSplitter.from_huggingface_tokenizer(tokenizer, capacity=capacity)
        self.collection_name: str = "embedded_articles"
        self.embedding = OpenAIEmbeddings(model="text-embedding-3-small")
        self.persist_dir: str = "../data/chroma_db"

        
    def chunk_documents(self, documents) -> List[Document]:
        chunked_documents = []
        for doc in documents:
            chunks = self.splitter.chunks(doc.page_content)
            for i, chunk in enumerate(chunks):
                chunked_documents.append(
                    Document(
                        page_content=chunk,
                        metadata={
                            **doc.metadata,
                            "id":          f"{doc.metadata['id']}_chunk_{i}",
                            "chunk_index": i,
                            "parent_id":   doc.metadata.get("id"),
                        }
                    )
                )
        print(f"[INFO] Split {len(documents)} documents into {len(chunked_documents)} chunks.")
        return chunked_documents

    def embed_documents(self, documents: List[Any]):
        persist_path = Path(self.persist_dir)
        if persist_path.exists():
            print(f"[INFO] Existing database found at '{self.persist_dir}', overwriting...")
            shutil.rmtree(persist_path)
        persist_path.mkdir(parents=True, exist_ok=True)
        chunks = self.chunk_documents(documents)
        print(f"[INFO] Embedding and storing {len(chunks)} chunks...")
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embedding,
            persist_directory=self.persist_dir,
            collection_name=self.collection_name,
        )
        print(f"[INFO] Stored {self.vectorstore._collection.count()} chunks in Chroma at '{self.persist_dir}'")

In [6]:
pipeline = EmbeddingDocument()
final_chunked = pipeline.embed_documents(documents)


[INFO] Existing database found at '../data/chroma_db', overwriting...
[INFO] Split 440 documents into 3035 chunks.
[INFO] Embedding and storing 3035 chunks...
[INFO] Stored 3035 chunks in Chroma at '../data/chroma_db'


In [10]:
class VectorStoreEmbedding:
    def __init__(self):
        self.persist_dir = str = "../data/chroma_db"
        self.collection_name = str = "embedded_articles"
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.vectorstore = None
        print(f"[INFO] VectorStoreEmbedding ready")

    def load(self):
        self.vectorstore = Chroma(
            persist_directory=self.persist_dir,
            embedding_function=self.embeddings,
            collection_name=self.collection_name,
        )
        print(f"[INFO] Loaded {self.vectorstore._collection.count()} chunks from '{self.persist_dir}'")

    def _paraphrase_query(self, query_text: str) -> str:
        prompt = (
            "Rewrite the following search query to be more specific and descriptive "
            "for retrieving relevant travel news articles. Return only the rewritten query, nothing else.\n\n"
            f"Query: {query_text}"
        )
        response = self.llm.invoke([HumanMessage(content=prompt)])
        paraphrased = response.content.strip()
        print(f"[INFO] Paraphrased: '{paraphrased}'")
        return paraphrased

    def query(self, query_text: str, top_k: int = 5) -> List[Document]:
        if self.vectorstore is None:
            raise RuntimeError("No vectorstore loaded. Call load() first.")
        paraphrased = self._paraphrase_query(query_text)
        return self.vectorstore.similarity_search(paraphrased, k=top_k)

In [22]:
def generate_response(query_text: str, top_k: int = 20) -> str:
    retriever = VectorStoreEmbedding()
    retriever.load()

    chunks: List[Document] = retriever.query(query_text, top_k=top_k)

    if not chunks:
        return "Sorry, I couldn't find any relevant articles."

    context = "\n\n".join(
        f"[Source: {doc.metadata.get('title', 'Unknown')} | {doc.metadata.get('publish_date', '')}]\n{doc.page_content}"
        for doc in chunks
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the user's question using only the provided context. "
                "If the context does not contain enough information, say so. "
                "Cite the article titles where relevant.\n\n"
                f"Context:\n{context}"
            )
        },
        {
            "role": "user",
            "content": query_text
        }
    ]

    response = retriever.llm.invoke(messages)
    print(f"[INFO] Generated response from {len(chunks)} chunks.")
    return response.content


In [23]:
response = generate_response("Travel destinations for US travelers in 2026")
print(response)


[INFO] VectorStoreEmbedding ready
[INFO] Loaded 3035 chunks from '../data/chroma_db'
[INFO] Paraphrased: 'Best travel destinations for US travelers in 2026 with a focus on emerging trends, safety, and unique experiences.'
[INFO] Generated response from 20 chunks.
In 2026, the top travel destinations for U.S. travelers include:

1. **Iceland** - Known for its geothermal springs, glaciers, and volcanic landscapes, with Reykjavik as a key city to explore.
2. **Switzerland** - Famous for its majestic Alps and luxury resorts, appealing to those seeking adventure and relaxation.
3. **Turkey** - Offers rich culture and history, with stunning architecture in Istanbul and unique landscapes in Cappadocia.
4. **Ireland** - Attracts visitors with its lush landscapes, historic castles, and vibrant pub culture, particularly in cities like Dublin and Galway.
5. **Denmark (Greenland)** - A remote destination known for its natural beauty, glaciers, and eco-tourism initiatives.
6. **Colombia** - Gaining

In [128]:
def _on_rm_error(func, path, exc_info):
    """Fix permissions and retry deletion."""
    os.chmod(path, stat.S_IWRITE)
    func(path)

chroma_path = "../data/chroma_db"
if Path(chroma_path).exists():
    shutil.rmtree(chroma_path, onerror=_on_rm_error)
    print(f"[INFO] Deleted existing Chroma DB at '{chroma_path}'")